In [2]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [3]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['easystats/parameters',
    'ds4dm/Tulip.jl',
    'mtiller/ModelicaBook',
    'jdtuck/fdasrvf_R',
    'xbarin02/collatz',
    'Quantum-TII/qibo',
    'jklimke/libcitygml',
    'mskcc/vcf2maf',
    'sibrun/GH',
    'lyc102/ifem']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0000296a2d6ee34cf9e717583df06a38e7842062,easystats_parameters


In [4]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  8%|████████████▊                                                                                                                                            | 267/3180 [04:32<49:33,  1.02s/it]

Got Errors {'4d050442e2c33da06a5ee4894fb01227c465a4b6': 'Key 4d050442e2c33da06a5ee4894fb01227c465a4b6 not found in /da5_fast/All.sha1c/commit_77.tch'}


 29%|████████████████████████████████████████████▎                                                                                                            | 922/3180 [15:54<39:20,  1.05s/it]

Got Errors {'4764a83171f6dd6363af12400381084140e1a168': 'Key 4764a83171f6dd6363af12400381084140e1a168 not found in /da5_fast/All.sha1c/commit_71.tch'}


 29%|████████████████████████████████████████████▋                                                                                                            | 930/3180 [16:03<39:07,  1.04s/it]

...

Got Errors {'c899646e406d33fae408416178c59884daef2302': 'Key c899646e406d33fae408416178c59884daef2302 not found in /da5_fast/All.sha1c/commit_72.tch'}


 94%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍         | 2981/3180 [51:40<03:27,  1.04s/it]

Got Errors {'d5a73fac8a6a9568f5a25f2358f2e61c82159177': 'Key d5a73fac8a6a9568f5a25f2358f2e61c82159177 not found in /da5_fast/All.sha1c/commit_85.tch'}


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3180/3180 [55:09<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0000296a2d6ee34cf9e717583df06a38e7842062,ffe4f603197ede26c59e16c6e791784e5da716e0,[2e3fdcdc4193569ac16e27e88bb3462870db3b97],Daniel <mail@danielluedecke.de>,1684657123,+0200,Daniel <mail@danielluedecke.de>,1684657123,+0200,Bug: `parameters` no longer respects the `vcov...,0000296a2d6ee34cf9e717583df06a38e7842062,easystats_parameters
1,000349ee3bbfe4b011a29a30f6d47a7a9486f5f6,d8e25e3c16d7ce476b7511e63dd79319529cd7b7,[dd955429c028f21baee6aa40160be1ae6653b4b6],GitHub Actions <actions@github.com>,1616069558,+0000,GitHub Actions <actions@github.com>,1616069558,+0000,Built site for parameters: 0.12.0.1@14c1600\n,000349ee3bbfe4b011a29a30f6d47a7a9486f5f6,easystats_parameters


In [5]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0000296a2d6ee34cf9e717583df06a38e7842062', 'tree': 'ffe4f603197ede26c59e16c6e791784e5da716e0', 'parent': ['2e3fdcdc4193569ac16e27e88bb3462870db3b97'], 'author': 'Daniel <mail@danielluedecke.de>', 'author_time': 1684657123, 'author_tz': '+0200', 'committer': 'Daniel <mail@danielluedecke.de>', 'committer_time': 1684657123, 'committer_tz': '+0200', 'message': 'Bug: `parameters` no longer respects the `vcov` argument\nFixes #881\n'}


In [6]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [7]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,lyc102_ifem,0000296a2d6ee34cf9e717583df06a38e7842062,Daniel <mail@danielluedecke.de>,1684657123,Bug: `parameters` no longer respects the `vcov...


In [8]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

Repo Name: easystats_parameters

Stars: 499

Forks: 46

Last Commit Date: 10/5/2026



Repo Name: ds4dm_tulip.jl

Stars: 163

Forks: 24

Last Commit Date: April 19, 2026



Repo Name: mtiller_modelicabook

Stars: 117

Forks: 75

Last Commit Date: September 6, 2026



Repo Name: jdtuck_fdasrvf_r

Stars: 15

Forks: 18

Last Commit Date: September 30, 2026



Repo Name: xbarin02_collatz

Stars: 84

Forks: 7

Last Commit Date: October 5, 2026



Repo Name: quantum-tii_qibo

Stars: 366

Forks: 104

Last Commit Date: October 7, 2026



Repo Name: jklimke_libcitygml

Stars: 111

Forks: 59

Last Commit Date: November 25, 2025



Repo Name: mskcc_vcf2maf

Stars: 421

Forks: 219 

Last Commit Date: May 25, 2026



Repo Name: sibrun_gh

Stars: 2

Forks: 3

Last Commit Date: March 11, 2026



Repo Name: lyc102_ifem

Stars: 297

Forks: 99

Last Commit Date: August 19, 2026


In [13]:
import pandas as pd

for project in df_commit_data["project"].unique():
    project_df = df_commit_data[df_commit_data["project"] == project]

    num_commits = project_df["commit"].nunique()
    num_authors = project_df["author"].nunique()

    min_time = project_df["author_time"].min()
    max_time = project_df["author_time"].max()

    print("Project:", project)
    print("Number of commits:", num_commits)
    print("Number of authors:", num_authors)
    print("Min time:", min_time)
    print("Max time:", max_time)
    print()

Project: easystats_parameters
Number of commits: 8925
Number of authors: 76
Min time: 1549684900
Max time: 1776169631

Project: ds4dm_tulip.jl
Number of commits: 926
Number of authors: 29
Min time: 1524835025
Max time: 1759338898

Project: mtiller_modelicabook
Number of commits: 2558
Number of authors: 69
Min time: 1354808010
Max time: 1742473600

Project: jdtuck_fdasrvf_r
Number of commits: 891
Number of authors: 25
Min time: 1347235093
Max time: 1760872097

Project: xbarin02_collatz
Number of commits: 1462
Number of authors: 4
Min time: 1564771496
Max time: 1762345592

Project: quantum-tii_qibo
Number of commits: 14496
Number of authors: 137
Min time: 1582014071
Max time: 1762323672

Project: jklimke_libcitygml
Number of commits: 598
Number of authors: 58
Min time: 1349083662
Max time: 1760908890

Project: mskcc_vcf2maf
Number of commits: 498
Number of authors: 42
Min time: 1385144375
Max time: 1754097415

Project: sibrun_gh
Number of commits: 1032
Number of authors: 11
Min time: 151